# Claude API의 프로그래매틱 도구 호출(PTC)

프로그래매틱 도구 호출(Programmatic Tool Calling, PTC)을 사용하면, 도구를 호출할 때마다 모델을 거쳐 왕복하는 대신 Claude가 코드 실행 환경 안에서 도구를 호출하는 코드를 직접 작성할 수 있습니다. 덕분에 여러 도구를 호출할 때의 종단 간 지연 시간이 크게 줄어들고, 관련 없는 내용이 모델의 컨텍스트 윈도에 도달하기 전에 걸러 내는 코드를 모델이 작성할 수 있어 토큰 소비도 극적으로 줄일 수 있습니다(예: 크고 잡음이 많은 파일에서 핵심 정보만 grep으로 찾아내기).

직접 수정할 수 없는 서드파티 API나 도구를 다뤄야 할 때, PTC는 코드 실행 환경에서 실행될 코드를 Claude가 작성하게 함으로써 컨텍스트 사용량을 줄이는 데 도움이 됩니다.

이 쿡북에서는 팀 경비 관리용 목 API를 다룹니다. 이 API는 여러 번 호출해야 하고 큰 결과를 반환하도록 설계되어 있어, 프로그래매틱 도구 호출의 이점을 보여 주기에 적합합니다.

## 이 쿡북을 마치면 다음을 할 수 있습니다.

- 일반 도구 호출과 프로그래매틱 도구 호출(PTC)의 차이 이해하기
- PTC를 활용하는 에이전트 작성하기

## 사전 준비

이 가이드를 따라 하기 전에 다음을 확인하세요.

**필요한 사전 지식**

- Python 기초 — async/await, 함수, 기본 자료구조에 익숙할 것
- 에이전트 패턴과 도구 호출에 대한 기본 이해

**필요한 도구**

- Python 3.11 이상
- Anthropic API 키
- Anthropic Python SDK 0.72 이상

## 준비

먼저 필요한 의존성을 설치합니다:

In [ ]:
# %pip install -r requirements.txt

참고: .env 파일에 다음 내용이 들어 있는지 확인하세요.

`ANTHROPIC_API_KEY=your_key_here`

환경 변수를 불러오고 클라이언트를 설정합니다. Claude 메시지 응답을 시각화하는 헬퍼 유틸리티도 함께 불러옵니다.

In [2]:
from dotenv import load_dotenv
from utils.visualize import visualize

load_dotenv()

MODEL = "claude-sonnet-4-6"

viz = visualize(auto_show=True)

## 서드파티 API 이해하기

[utils/team_expense_api.py](utils/team_expense_api.py)에는 `get_team_members`, `get_expenses`, `get_custom_budget` 세 함수가 정의되어 있습니다. `get_team_members`는 특정 부서의 모든 직원을 역할, 직급, 연락처 정보와 함께 조회합니다. `get_expenses`는 특정 분기의 직원별 경비 항목을 모두 반환하는데, 직원 한 명당 수백 건에 이를 수 있고 각 항목에는 영수증 URL, 결재 라인, 가맹점 정보 등 방대한 메타데이터가 담겨 있습니다. `get_custom_budget`은 해당 직원에게 별도의 출장비 예외 한도가 적용되는지 확인합니다(적용되지 않으면 분기당 표준 한도인 $5,000을 사용합니다).

이 시나리오에서는 팀 경비를 분석해 어떤 직원이 예산을 초과했는지 찾아내야 합니다. 전통적인 방식이라면 사람마다 경비 보고서를 직접 뽑아 항목별로 합산하고, 예산 한도와 비교하고(예외 한도가 있는지도 확인하고), 보고서를 정리해야 할 것입니다. 대신 여기서는 Claude에게 이 분석을 맡깁니다. 사용 가능한 도구로 팀 데이터를 조회하고, 풍부한 메타데이터가 붙은 수백 건의 경비 항목을 가져와, 누가 예산을 초과했는지 판단하게 합니다.

여기서의 핵심 난제는 직원 한 명당 100건이 넘는 경비 항목을 가져와 파싱하고 집계해야 한다는 점, 그리고 `get_custom_budget` 도구는 경비를 분석해 표준 예산을 초과한 사람을 가려낸 뒤에야 호출할 수 있다는 점입니다. 이렇게 순차적인 의존 관계가 생기기 때문에, 프로그래매틱 도구 호출의 이점을 보여 주기에 이상적인 사례가 됩니다.

도구 정의를 messages API에 전달하고 Claude에 분석을 요청하겠습니다. Claude API에서 도구 사용이 어떻게 동작하는지 익숙하지 않다면 [도구 사용 구현하기](https://docs.claude.com/en/docs/agents-and-tools/tool-use/implement-tool-use) 문서를 읽어 보세요.

In [3]:
import json

import anthropic
from utils.team_expense_api import get_custom_budget, get_expenses, get_team_members

client = anthropic.Anthropic()

# Tool definitions for the team expense API
tools = [
    {
        "name": "get_team_members",
        "description": 'Returns a list of team members for a given department. Each team member includes their ID, name, role, level (junior, mid, senior, staff, principal), and contact information. Use this to get a list of people whose expenses you want to analyze. Available departments are: engineering, sales, and marketing.\n\nRETURN FORMAT: Returns a JSON string containing an ARRAY of team member objects (not wrapped in an outer object). Parse with json.loads() to get a list. Example: [{"id": "ENG001", "name": "Alice", ...}, {"id": "ENG002", ...}]',
        "input_schema": {
            "type": "object",
            "properties": {
                "department": {
                    "type": "string",
                    "description": "The department name. Case-insensitive.",
                }
            },
            "required": ["department"],
        },
        "input_examples": [
            {"department": "engineering"},
            {"department": "sales"},
            {"department": "marketing"},
        ],
    },
    {
        "name": "get_expenses",
        "description": "Returns all expense line items for a given employee in a specific quarter. Each expense includes extensive metadata: date, category, description, amount (in USD), currency, status (approved, pending, rejected), receipt URL, approval chain, merchant name and location, payment method, and project codes. An employee may have 20-50+ expense line items per quarter, and each line item contains substantial metadata for audit and compliance purposes. Categories include: 'travel' (flights, trains, rental cars, taxis, parking), 'lodging' (hotels, airbnb), 'meals', 'software', 'equipment', 'conference', 'office', and 'internet'. IMPORTANT: Only expenses with status='approved' should be counted toward budget limits.\n\nRETURN FORMAT: Returns a JSON string containing an ARRAY of expense objects (not wrapped in an outer object with an 'expenses' key). Parse with json.loads() to get a list directly. Example: [{\"expense_id\": \"ENG001_Q3_001\", \"amount\": 1250.50, \"category\": \"travel\", ...}, {...}]",
        "input_schema": {
            "type": "object",
            "properties": {
                "employee_id": {
                    "type": "string",
                    "description": "The unique employee identifier",
                },
                "quarter": {
                    "type": "string",
                    "description": "Quarter identifier: 'Q1', 'Q2', 'Q3', or 'Q4'",
                },
            },
            "required": ["employee_id", "quarter"],
        },
        "input_examples": [
            {"employee_id": "ENG001", "quarter": "Q3"},
            {"employee_id": "SAL002", "quarter": "Q1"},
            {"employee_id": "MKT001", "quarter": "Q4"},
        ],
    },
    {
        "name": "get_custom_budget",
        "description": 'Get the custom quarterly travel budget for a specific employee. Most employees have a standard $5,000 quarterly travel budget. However, some employees have custom budget exceptions based on their role requirements. This function checks if a specific employee has a custom budget assigned.\n\nRETURN FORMAT: Returns a JSON string containing a SINGLE OBJECT (not an array). Parse with json.loads() to get a dict. Example: {"user_id": "ENG001", "has_custom_budget": false, "travel_budget": 5000, "reason": "Standard", "currency": "USD"}',
        "input_schema": {
            "type": "object",
            "properties": {
                "user_id": {
                    "type": "string",
                    "description": "The unique employee identifier",
                }
            },
            "required": ["user_id"],
        },
        "input_examples": [
            {"user_id": "ENG001"},
            {"user_id": "SAL002"},
            {"user_id": "MKT001"},
        ],
    },
]

tool_functions = {
    "get_team_members": get_team_members,
    "get_expenses": get_expenses,
    "get_custom_budget": get_custom_budget,
}

## 전통적 도구 호출 (기준점)

첫 번째 예제에서는 전통적 도구 호출로 기준점을 잡겠습니다.

초기 질의로 `messages.create` API를 호출합니다. 모델이 `tool_use` 사유로 멈추면 요청받은 도구를 실행하고, 그 출력을 메시지에 추가한 뒤 모델을 다시 호출합니다.

In [4]:
import time

from anthropic.types import TextBlock, ToolUseBlock
from anthropic.types.beta import (
    BetaMessageParam as MessageParam,
)
from anthropic.types.beta import (
    BetaTextBlock,
    BetaToolUseBlock,
)

messages: list[MessageParam] = []


def run_agent_without_ptc(user_message):
    """Run agent using traditional tool calling"""
    messages.append({"role": "user", "content": user_message})
    total_tokens = 0
    start_time = time.time()
    api_counter = 0

    while True:
        response = client.beta.messages.create(
            model=MODEL,
            max_tokens=4000,
            tools=tools,
            messages=messages,
            betas=["advanced-tool-use-2025-11-20"],
        )

        api_counter += 1

        # Track token usage
        total_tokens += response.usage.input_tokens + response.usage.output_tokens
        viz.capture(response)
        if response.stop_reason == "end_turn":
            # Extract the first text block from the response
            final_response = next(
                (
                    block.text
                    for block in response.content
                    if isinstance(block, (BetaTextBlock, TextBlock))
                ),
                None,
            )
            elapsed_time = time.time() - start_time
            return final_response, messages, total_tokens, elapsed_time, api_counter

        # Process tool calls
        if response.stop_reason == "tool_use":
            # First, add the assistant's response to messages
            messages.append({"role": "assistant", "content": response.content})

            # Collect all tool results
            tool_results = []

            for block in response.content:
                if isinstance(block, (BetaToolUseBlock, ToolUseBlock)):
                    tool_name = block.name
                    tool_input = block.input
                    tool_use_id = block.id

                    result = tool_functions[tool_name](**tool_input)

                    content = str(result)

                    tool_result = {
                        "type": "tool_result",
                        "tool_use_id": tool_use_id,
                        "content": content,
                    }
                    tool_results.append(tool_result)

            # Append all tool results at once after collecting them
            messages.append({"role": "user", "content": tool_results})

        else:
            print(f"\nUnexpected stop reason: {response.stop_reason}")
            elapsed_time = time.time() - start_time

            final_response = next(
                (
                    block.text
                    for block in response.content
                    if isinstance(block, (BetaTextBlock, TextBlock))
                ),
                f"Stopped with reason: {response.stop_reason}",
            )
            return final_response, messages, total_tokens, elapsed_time, api_counter

모델에 보내는 초기 질의에는 모델을 안내하는 지시가 일부 포함되어 있습니다. 간결함을 위해 각 도구를 한 번씩만 호출하도록 요청했습니다. 더 깊이 조사해야 하는 경우라면 모델이 여러 시스템이나 여러 기간을 들여다보려 할 수도 있습니다.

In [5]:
query = "Which engineering team members exceeded their Q3 travel budget? Standard quarterly travel budget is $5,000. However, some employees have custom budget limits. For anyone who exceeded the $5,000 standard budget, check if they have a custom budget exception. If they do, use that custom limit instead to determine if they truly exceeded their budget."

In [6]:
# Run the agent
result, conversation, total_tokens, elapsed_time, api_count_without_ptc = run_agent_without_ptc(
    query
)

print(f"Result: {result}")
print(f"API calls made: {api_count_without_ptc}")
print(f"Total tokens used: {total_tokens:,}")
print(f"Total time taken: {elapsed_time:.2f}s")

╭────────────────────────────────────────────── Claude API Response ──────────────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 1,859 in • 85 out • 1,944 total                                            │
│ ├── Model: claude-sonnet-4-6                                                                           │
│ ├── Stop Reason: tool_use                                                                                       │
│ └── Content (2 blocks)                                                                                          │
│     ├── Block 1                                                                                                 │
│     │   └── Text                                                                                                │
│     │       └── I'll help you identify which engineering team members exceeded their Q3 travel budget. Let me   │
│     │           start by getting the list of engineering team members.                                          │
│     └── Block 2                                                                                                 │
│         └── Tool Use: get_team_members                                                                          │
│             ├── ID: toolu_01LuouuJYp1sSvBe2Du7EG7v                                                              │
│             ├── Caller: model (direct)                                                                          │
│             └── Input:                                                                                          │
│                 └── {                                                                                           │
│                       "department": "engineering"                                                               │
│                     }                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── Claude API Response ───────────────────────────────╮
│ Claude Message (assistant) │ tokens: 2,473 in • 497 out • 2,970 total            │
│ ├── Model: claude-sonnet-4-6                                            │
│ ├── Stop Reason: tool_use                                                        │
│ └── Content (9 blocks)                                                           │
│     ├── Block 1                                                                  │
│     │   └── Text                                                                 │
│     │       └── Now let me get the Q3 expenses for all engineering team members: │
│     ├── Block 2                                                                  │
│     │   └── Tool Use: get_expenses                                               │
│     │       ├── ID: toolu_01Wu8LLTT2sKTTqpVwGT65Lj                               │
│     │       ├── Caller: model (direct)                                           │
│     │       └── Input:                                                           │
│     │           └── {                                                            │
│     │                 "employee_id": "ENG001",                                   │
│     │                 "quarter": "Q3"                                            │
│     │               }                                                            │
│     ├── Block 3                                                                  │
│     │   └── Tool Use: get_expenses                                               │
│     │       ├── ID: toolu_01KzjQ5mQJa9ocWjCGzYkD9F                               │
│     │       ├── Caller: model (direct)                                           │
│     │       └── Input:                                                           │
│     │           └── {                                                            │
│     │                 "employee_id": "ENG002",                                   │
│     │                 "quarter": "Q3"                                            │
│     │               }                                                            │
│     ├── Block 4                                                                  │
│     │   └── Tool Use: get_expenses                                               │
│     │       ├── ID: toolu_01RjjhZTg9JsKXE5E9S6Foho                               │
│     │       ├── Caller: model (direct)                                           │
│     │       └── Input:                                                           │
│     │           └── {                                                            │
│     │                 "employee_id": "ENG003",                                   │
│     │                 "quarter": "Q3"                                            │
│     │               }                                                            │
│     ├── Block 5                                                                  │
│     │   └── Tool Use: get_expenses                                               │
│     │       ├── ID: toolu_013xqpxpfc2N9rP5W5uMLAo9                               │
│     │       ├── Caller: model (direct)                                           │
│     │       └── Input:                                                           │
│     │           └── {                                                            │
│     │                 "employee_id": "ENG004",                                   │
│     │                 "quarter": "Q3"                                            │
│     │               }                                                            │
│     ├── Block 6                                                                  │
│     │   └── Tool Use: get_expenses                                               │
│     │       ├── ID: toolu_019zfzG6Wox8iDqy1dUXiH3t                               │
│     │       

╭────────────────────────────────────────────── Claude API Response ──────────────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 51,744 in • 290 out • 52,034 total                                         │
│ ├── Model: claude-sonnet-4-6                                                                           │
│ ├── Stop Reason: tool_use                                                                                       │
│ └── Content (7 blocks)                                                                                          │
│     ├── Block 1                                                                                                 │
│     │   └── Text                                                                                                │
│     │       └── Now let me calculate the approved travel expenses for each engineer and identify who exceeded   │
│     │           $5,000:                                                                                         │
│     ├── Block 2                                                                                                 │
│     │   └── Tool Use: get_custom_budget                                                                         │
│     │       ├── ID: toolu_013oegKwjvToLwEW1daDD8av                                                              │
│     │       ├── Caller: model (direct)                                                                          │
│     │       └── Input:                                                                                          │
│     │           └── {                                                                                           │
│     │                 "user_id": "ENG001"                                                                       │
│     │               }                                                                                           │
│     ├── Block 3                                                                                                 │
│     │   └── Tool Use: get_custom_budget                                                                         │
│     │       ├── ID: toolu_0162W4Ycr9FcVVED65exjAj4                                                              │
│     │       ├── Caller: model (direct)                                                                          │
│     │       └── Input:                                                                                          │
│     │           └── {                                                                                           │
│     │                 "user_id": "ENG003"                                                                       │
│     │               }                                                                                           │
│     ├── Block 4                                                                                                 │
│     │   └── Tool Use: get_custom_budget                                                                         │
│     │       ├── ID: toolu_01JcTX5rnwFxA99Am33gXmh6                                                              │
│     │       ├── Caller: model (direct)                                                                          │
│     │       └── Input:                                                                                          │
│     │           └── {                                                                                           │
│     │                 "user_id": "ENG005"                                                                       │
│     │               }                                                                                           │
│     ├── Block 5                                                                                                 │
│     │   └── Tool Use: get_custom_budget                        

╭────────────────────────────────────────────── Claude API Response ──────────────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 52,533 in • 992 out • 53,525 total                                         │
│ ├── Model: claude-sonnet-4-6                                                                           │
│ ├── Stop Reason: end_turn                                                                                       │
│ └── Content (1 blocks)                                                                                          │
│     └── Block 1                                                                                                 │
│         └── Text                                                                                                │
│             └── Now let me analyze the data. I'll calculate the approved travel expenses for each engineer:     │
│                                                                                                                 │
│                 **Analysis of Q3 Travel Expenses:**                                                             │
│                                                                                                                 │
│                 **ENG001 - Alice Chen (Senior Software Engineer)**                                              │
│                 - Approved travel expenses: $1,161.04 + $18.63 + $13.21 + $36.55 + $1,440.42 + $166.46 + $48.43 │
│                 + $1,124.56 + $1,245.90 + $1,498.42 = **$6,753.62**                                             │
│                 - Budget: $5,000 (Standard)                                                                     │
│                 - **EXCEEDED by $1,753.62** ❌                                                                  │
│                                                                                                                 │
│                 **ENG002 - Bob Martinez (Staff Engineer)**                                                      │
│                 - Approved travel expenses: $180.16 + $10.07 + $20.76 = **$210.99**                             │
│                 - Budget: $5,000 (Standard)                                                                     │
│                 - Under budget ✓                                                                                │
│                                                                                                                 │
│                 **ENG003 - Carol White (Software Engineer)**                                                    │
│                 - Approved travel expenses: $24.75 + $424.74 + $1,397.17 + $1,026.12 + $1,288.36 + $1,128.90 +  │
│                 $1,148.42 + $45.03 = **$6,483.49**                                                              │
│                 - Budget: $5,000 (Standard)                                                                     │
│                 - **EXCEEDED by $1,483.49** ❌                                                                  │
│                                                                                                                 │
│                 **ENG004 - David Kim (Principal Engineer)**                                                     │
│                 - Approved travel expenses: $21.68 + $46.12 + $1,008.68 + $46.43 = **$1,122.91**                │
│                 - Budget: $5,000 (Standard)                                                                     │
│                 - Under budget ✓                                                                                │
│                                                                                                                 │
│                 **ENG005 - Emma Johnson (Junior Software Engineer)                                              │
│                 ... (truncated)                                  

Result: Now let me analyze the data. I'll calculate the approved travel expenses for each engineer:

**Analysis of Q3 Travel Expenses:**

**ENG001 - Alice Chen (Senior Software Engineer)**
- Approved travel expenses: $1,161.04 + $18.63 + $13.21 + $36.55 + $1,440.42 + $166.46 + $48.43 + $1,124.56 + $1,245.90 + $1,498.42 = **$6,753.62**
- Budget: $5,000 (Standard)
- **EXCEEDED by $1,753.62** ❌

**ENG002 - Bob Martinez (Staff Engineer)**
- Approved travel expenses: $180.16 + $10.07 + $20.76 = **$210.99**
- Budget: $5,000 (Standard)
- Under budget ✓

**ENG003 - Carol White (Software Engineer)**
- Approved travel expenses: $24.75 + $424.74 + $1,397.17 + $1,026.12 + $1,288.36 + $1,128.90 + $1,148.42 + $45.03 = **$6,483.49**
- Budget: $5,000 (Standard)
- **EXCEEDED by $1,483.49** ❌

**ENG004 - David Kim (Principal Engineer)**
- Approved travel expenses: $21.68 + $46.12 + $1,008.68 + $46.43 = **$1,122.91**
- Budget: $5,000 (Standard)
- Under budget ✓

**ENG005 - Emma Johnson (Junior Software E

좋습니다! Claude가 사용 가능한 도구를 제대로 활용해 출장비 예산을 초과한 팀원을 찾아낸 것을 볼 수 있습니다. 다만 이 작업에 토큰을 많이 썼다는 것도 확인할 수 있습니다. 항목을 파싱하고 범주별 합계를 내고 예산 한도와 비교하기 위해, Claude는 모든 경비 항목을 컨텍스트 윈도로 받아들여야 했습니다. 직원 한 명당 100건이 넘을 수 있고, 각 항목에는 영수증 URL, 결재 라인, 가맹점 정보 등 방대한 메타데이터가 붙어 있습니다.

게다가 전통적 도구 호출 방식은 여러 번의 순차적 왕복을 필요로 합니다. 먼저 팀원을 조회하고, 각자의 경비를 가져오고, 표준 한도를 넘긴 사람에 대해 예외 한도를 확인하는 식입니다. 왕복할 때마다 지연이 쌓이고, 경비 기록의 모든 메타데이터가 모델의 컨텍스트를 통과합니다.

이제 Claude가 코드 실행 환경에서 대용량 데이터를 처리하는 코드를 작성하게 하는 PTC로 성능을 개선할 수 있는지 살펴보겠습니다.

도구에 PTC를 활성화하려면, 코드 실행으로 호출할 수 있어야 하는 도구에 먼저 `allowed_callers` 필드를 추가해야 합니다.

**고려할 핵심 사항**

- `allowed_callers`가 없는 도구는 기본적으로 모델만 호출할 수 있습니다
- 호출자를 여러 개 지정하면 모델과 코드 실행 양쪽에서 호출할 수 있습니다: `["direct", "code_execution_20250825"]`
- 프로그래매틱하게 반복 실행해도 안전한 도구만 선택적으로 허용하세요.

In [7]:
import copy

ptc_tools = copy.deepcopy(tools)
for tool in ptc_tools:
    tool["allowed_callers"] = ["code_execution_20250825"]  # type: ignore


# Add the code execution tool
ptc_tools.append(
    {
        "type": "code_execution_20250825",  # type: ignore
        "name": "code_execution",
    }
)

도구 정의를 프로그래매틱 도구 호출이 가능하도록 수정했으니, 이제 PTC로 에이전트를 실행할 수 있습니다. 이를 위해 함수를 몇 가지 수정해야 했습니다. `beta` messages API를 사용해야 합니다.

1. betas에 `"advanced-tool-use-2025-11-20"`을 추가했습니다.
2. `container_id`가 정의되어 있으면 요청에 함께 전달합니다. 이는 이번처럼 상태를 유지하는 워크플로에서만 필요하며, 단일 턴 워크플로에서는 필요하지 않습니다.
3. `tool_use` 블록의 `caller` 필드를 확인하면 이 도구 호출이 모델의 직접 호출인지 프로그래매틱 호출인지 판단할 수 있습니다.

어느 쪽이든 도구 결과는 Claude API를 통해 보내지만, 모델이 "보게" 되는 것은 `direct` 호출뿐입니다. `code_execution_20250825` 유형은 코드 실행 컨테이너에서만 보입니다.

In [8]:
messages = []


def run_agent_with_ptc(user_message):
    """Run agent using PTC"""
    messages.append({"role": "user", "content": user_message})
    total_tokens = 0
    start_time = time.time()
    container_id = None
    api_counter = 0

    while True:
        # Build request with PTC beta headers
        request_params = {
            "model": MODEL,
            "max_tokens": 4000,
            "tools": ptc_tools,
            "messages": messages,
        }

        response = client.beta.messages.create(
            **request_params,
            betas=[
                "advanced-tool-use-2025-11-20",
            ],
            extra_body={"container": container_id} if container_id else None,
        )
        viz.capture(response)
        api_counter += 1

        # Track container for stateful execution
        if hasattr(response, "container") and response.container:
            container_id = response.container.id
            print(f"\n[Container] ID: {container_id}")
            if hasattr(response.container, "expires_at"):
                # If the container has expired, we would need to restart our workflow. In our case, it completes before expiration.
                print(f"[Container] Expires at: {response.container.expires_at}")

        # Track token usage
        total_tokens += response.usage.input_tokens + response.usage.output_tokens

        if response.stop_reason == "end_turn":
            # Extract the first text block from the response
            final_response = next(
                (block.text for block in response.content if isinstance(block, BetaTextBlock)),
                None,
            )
            elapsed_time = time.time() - start_time
            return final_response, messages, total_tokens, elapsed_time, api_counter

        # As before, we process tool calls
        if response.stop_reason == "tool_use":
            # First, add the assistant's response to messages
            messages.append({"role": "assistant", "content": response.content})

            # Collect all tool results
            tool_results = []

            for block in response.content:
                if isinstance(block, BetaToolUseBlock):
                    tool_name = block.name
                    tool_input = block.input
                    tool_use_id = block.id

                    # We can use caller type to understand how the tool was invoked
                    caller_type = block.caller["type"]  # type: ignore

                    if caller_type == "code_execution_20250825":
                        print(f"[PTC] Tool called from code execution environment: {tool_name}")

                    elif caller_type == "direct":
                        print(f"[Direct] Tool called by model: {tool_name}")

                    result = tool_functions[tool_name](**tool_input)

                    # Format result as proper content for the API
                    if isinstance(result, list) and result and isinstance(result[0], str):
                        content = "\n".join(result)
                    elif isinstance(result, (dict, list)):
                        content = json.dumps(result)
                    else:
                        content = str(result)

                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": tool_use_id,
                            "content": content,
                        }
                    )

            messages.append({"role": "user", "content": tool_results})

        else:
            print(f"\nUnexpected stop reason: {response.stop_reason}")
            elapsed_time = time.time() - start_time

            final_response = next(
                (block.text for block in response.content if isinstance(block, BetaTextBlock)),
                f"Stopped with reason: {response.stop_reason}",
            )
            return final_response, messages, total_tokens, elapsed_time, api_counter

In [9]:
# Run the PTC agent
result_ptc, conversation_ptc, total_tokens_ptc, elapsed_time_ptc, api_count_with_ptc = (
    run_agent_with_ptc(query)
)

╭────────────────────────────────────────────── Claude API Response ──────────────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 4,134 in • 539 out • 4,673 total                                           │
│ ├── Model: claude-sonnet-4-6                                                                           │
│ ├── Stop Reason: tool_use                                                                                       │
│ └── Content (3 blocks)                                                                                          │
│     ├── Block 1                                                                                                 │
│     │   └── Text                                                                                                │
│     │       └── I'll help you identify which engineering team members exceeded their Q3 travel budget. Let me   │
│     │           start by getting the engineering team members and their expenses.                               │
│     ├── Block 2                                                                                                 │
│     │   └── Server Tool Use                                                                                     │
│     │       ├── ID: srvtoolu_015mWPqaFni4B313UieCxbny                                                           │
│     │       ├── Caller: direct                                                                                  │
│     │       └── Code:                                                                                           │
│     │           └──    1                                                                                        │
│     │                  2 import asyncio                                                                         │
│     │                  3 import json                                                                            │
│     │                  4                                                                                        │
│     │                  5 async def main():                                                                      │
│     │                  6     # First, get all engineering team members                                          │
│     │                  7     team_members_json = await get_team_members({'department': 'engineering'})          │
│     │                  8     team_members = json.loads(team_members_json)                                       │
│     │                  9                                                                                        │
│     │                 10     print(f"Found {len(team_members)} engineering team members")                       │
│     │                 11                                                                                        │
│     │                 12     # Get Q3 expenses for all team members in parallel                                 │
│     │                 13     expense_tasks = [                                                                  │
│     │                 14         get_expenses({'employee_id': member['id'], 'quarter': 'Q3'})                   │
│     │                 15         for member in team_members                                                     │
│     │                 16     ]                                                                                  │
│     │                 17                                                                                        │
│     │                 18     expenses_results = await asyncio.gather(*expense_tasks)                            │
│     │                 19                                                                                        │
│     │                 20     # Calculate travel expenses for each member                                        │
│     │                 21     travel_spending = {}              


[Container] ID: container_011CVSAwq5J4vNPi3A4P2Rwh
[Container] Expires at: 2025-11-24 05:41:17.467494+00:00
[PTC] Tool called from code execution environment: get_team_members


╭──────────────────── Claude API Response ────────────────────╮
│ Claude Message (assistant) │ tokens: 0 in • 0 out • 0 total │
│ ├── Model: claude-sonnet-4-6                       │
│ ├── Stop Reason: tool_use                                   │
│ └── Content (8 blocks)                                      │
│     ├── Block 1                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01Nq2au3W69RmDFZaSdqe6u1          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG007",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 2                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01YYw9cuTSXk7bu7P38qBz6P          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG005",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 3                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01Fyxxe2KmVpVmw4jJL2CXSz          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG008",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 4                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01J4ovDu2UJa9Se19vxKTa6y          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG006",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 5                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01T24CrvQYA3LqGfZftCmueC          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG003",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 6                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01HotYZN6sk3gMLkpdWXbdz4          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG004",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 7                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01AxvEqi3AKqdnH44kGH1U6E          │
│     │       ├── Caller: code execution 


[Container] ID: container_011CVSAwq5J4vNPi3A4P2Rwh
[Container] Expires at: 2025-11-24 05:41:19.266670+00:00
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses


╭─────────────────────────────────────── Claude API Response ───────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 4,751 in • 679 out • 5,430 total                             │
│ ├── Model: claude-sonnet-4-6                                                             │
│ ├── Stop Reason: tool_use                                                                         │
│ └── Content (6 blocks)                                                                            │
│     ├── Block 1                                                                                   │
│     │   └── Code Execution Result: Success (exit 0)                                               │
│     │       └── stdout:                                                                           │
│     │           └── Found 8 engineering team members                                              │
│     │                                                                                             │
│     │               Employees who exceeded $5,000 standard budget: 3                              │
│     │               ENG001: Alice Chen - $9177.88                                                 │
│     │               ENG003: Carol White - $6483.49                                                │
│     │               ENG007: Grace Taylor - $5289.35                                               │
│     │                                                                                             │
│     ├── Block 2                                                                                   │
│     │   └── Text                                                                                  │
│     │       └── Now let me check if any of these three employees have custom budget exceptions:   │
│     ├── Block 3                                                                                   │
│     │   └── Server Tool Use                                                                       │
│     │       ├── ID: srvtoolu_015GTpFmCbd2JPAQLAioB4Qb                                             │
│     │       ├── Caller: direct                                                                    │
│     │       └── Code:                                                                             │
│     │           └──    1                                                                          │
│     │                  2 import asyncio                                                           │
│     │                  3 import json                                                              │
│     │                  4                                                                          │
│     │                  5 async def main():                                                        │
│     │                  6     # Check custom budgets for the three employees who exceeded standard │
│     │                  7     exceeded_ids = ['ENG001', 'ENG003', 'ENG007']                        │
│     │                  8     exceeded_amounts = {                                                 │
│     │                  9         'ENG001': {'name': 'Alice Chen', 'travel_total': 9177.88},       │
│     │                 10         'ENG003': {'name': 'Carol White', 'travel_total': 6483.49},      │
│     │                 11         'ENG007': {'name': 'Grace Taylor', 'travel_total': 5289.35}      │
│     │                 12     }                                                                    │
│     │                 13                                                                          │
│     │                 14     # Get custom budgets in parallel                                     │
│     │                 15     budget_tasks = [                                                     │
│     │                 16         get_custom_budget({'user_id': emp_id})                           │
│     │                 17     


[Container] ID: container_011CVSAwq5J4vNPi3A4P2Rwh
[Container] Expires at: 2025-11-24 05:41:33.430636+00:00
[PTC] Tool called from code execution environment: get_custom_budget
[PTC] Tool called from code execution environment: get_custom_budget
[PTC] Tool called from code execution environment: get_custom_budget


╭────────────────────────────────────────────── Claude API Response ──────────────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 5,611 in • 205 out • 5,816 total                                           │
│ ├── Model: claude-sonnet-4-6                                                                           │
│ ├── Stop Reason: end_turn                                                                                       │
│ └── Content (2 blocks)                                                                                          │
│     ├── Block 1                                                                                                 │
│     │   └── Code Execution Result: Success (exit 0)                                                             │
│     │       └── stdout:                                                                                         │
│     │           └── ENGINEERING TEAM MEMBERS WHO EXCEEDED THEIR Q3 TRAVEL BUDGET:                               │
│     │               ================================================================================            │
│     │                                                                                                           │
│     │               Alice Chen (ENG001)                                                                         │
│     │                 Budget Limit: $5,000.00 (Standard)                                                        │
│     │                 Travel Spending: $9,177.88                                                                │
│     │                 Exceeded By: $4,177.88                                                                    │
│     │                                                                                                           │
│     │               Carol White (ENG003)                                                                        │
│     │                 Budget Limit: $5,000.00 (Standard)                                                        │
│     │                 Travel Spending: $6,483.49                                                                │
│     │                 Exceeded By: $1,483.49                                                                    │
│     │                                                                                                           │
│     │               Grace Taylor (ENG007)                                                                       │
│     │                 Budget Limit: $5,000.00 (Standard)                                                        │
│     │                 Travel Spending: $5,289.35                                                                │
│     │                 Exceeded By: $289.35                                                                      │
│     │                                                                                                           │
│     └── Block 2                                                                                                 │
│         └── Text                                                                                                │
│             └── ## Summary                                                                                      │
│                                                                                                                 │
│                 **Three engineering team members exceeded their Q3 travel budget:**                             │
│                                                                                                                 │
│                 1. **Alice Chen (ENG001)**                                                                      │
│                    - Budget: $5,000 (Standard)                                                                  │
│                    - Spent: $9,177.88                          

In [10]:
print(f"\n{'=' * 60}")
print(f"Result: {result_ptc}")
print(f"\n{'=' * 60}")
print("Performance Metrics:")
print(
    f"  Total API calls to Claude: {len([m for m in conversation_ptc if m['role'] == 'assistant'])}"
)
print(f"  Total tokens used: {total_tokens_ptc:,}")
print(f"  Total time taken: {elapsed_time_ptc:.2f}s")


Result: ## Summary

**Three engineering team members exceeded their Q3 travel budget:**

1. **Alice Chen (ENG001)**
   - Budget: $5,000 (Standard)
   - Spent: $9,177.88
   - Over budget by: **$4,177.88**

2. **Carol White (ENG003)**
   - Budget: $5,000 (Standard)
   - Spent: $6,483.49
   - Over budget by: **$1,483.49**

3. **Grace Taylor (ENG007)**
   - Budget: $5,000 (Standard)
   - Spent: $5,289.35
   - Over budget by: **$289.35**

All three employees are on the standard $5,000 quarterly travel budget with no custom exceptions, so they all genuinely exceeded their allocated travel budget for Q3.

Performance Metrics:
  Total API calls to Claude: 3
  Total tokens used: 15,919
  Total time taken: 34.88s


## 성능 비교

전통적 도구 호출과 PTC의 성능을 비교해 보겠습니다.

**API 호출 횟수에 대한 참고:** 이 예제에서는 PTC 쪽이 API 호출을 더 많이 하는 것을 볼 수 있습니다. PTC가 모범 사례를 따르는 더 구조적이고 순차적인 코드를 작성하기 때문입니다. 예를 들어 경비 조회 단계와 예산 확인 단계를 분리합니다. 전통적 도구 호출은 여러 작업을 한 턴에 묶을 수 있을 때도 있지만, 그 대가로 모든 원시 데이터가 모델의 컨텍스트를 통과합니다. 특히 메타데이터가 풍부한 대용량 데이터셋을 다룰 때는 PTC의 토큰 효율 이득이 왕복 횟수가 조금 늘어나는 비용을 크게 웃돕니다.

In [11]:
import pandas as pd

# Create comparison dataframe
comparison_data = {
    "Metric": [
        "API Calls",
        "Total Tokens",
        "Elapsed Time (s)",
        "Token Reduction",
        "Time Reduction",
    ],
    "Traditional": [
        api_count_without_ptc,
        f"{total_tokens:,}",
        f"{elapsed_time:.2f}",
        "-",
        "-",
    ],
    "PTC": [
        api_count_with_ptc,
        f"{total_tokens_ptc:,}",
        f"{elapsed_time_ptc:.2f}",
        f"{((total_tokens - total_tokens_ptc) / total_tokens * 100):.1f}%",
        f"{((elapsed_time - elapsed_time_ptc) / elapsed_time * 100):.1f}%",
    ],
}

df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))

          Metric Traditional    PTC
       API Calls           4      4
    Total Tokens     110,473 15,919
Elapsed Time (s)       35.38  34.88
 Token Reduction           -  85.6%
  Time Reduction           -   1.4%


## 핵심 정리

이 예제에서 PTC는 세 가지 핵심 역량을 통해 상당한 성능 개선을 보여 주었습니다.

### 1. 대용량 데이터 파싱을 통한 컨텍스트 보존
이번 워크플로에서 드러난 가장 큰 이점입니다. Claude는 코드 실행 환경 안에서 수백 건의 경비 항목을 가져와 처리하는 코드를 작성했습니다. 데이터를 프로그래매틱하게 처리함으로써 JSON을 파싱하고, 상태로 필터링하고, 범주별로 금액을 합산하고, 예산 한도와 비교하는 일을 모두 원시 경비 데이터와 메타데이터를 모델의 컨텍스트 윈도로 보내지 않고 해냈습니다. 그 결과 **토큰 사용량이 크게 줄었습니다**.

### 2. 순차적 의존 관계 최적화
이 API에는 순차적 의존 관계가 있습니다. `get_custom_budget(user_id)`는 경비를 분석해 표준 $5,000 예산을 초과한 사람을 가려낸 뒤에야 호출할 수 있습니다. 전통적 도구 호출에서는 팀원 조회, 각자의 경비 조회, 초과자 식별, 그리고 한 명씩 예외 한도 확인이라는 여러 번의 왕복이 필요합니다. PTC에서는 Claude가 이 워크플로 전체를 코드 실행 환경에서 오케스트레이션하는 코드를 작성해, 반복문 안에서 프로그래매틱 도구 호출을 하고 호출 간 상태를 유지합니다. 여러 번의 순차적 API 왕복이 더 적은 횟수의 똑똑한 오케스트레이션으로 바뀝니다.

### 3. 코드 실행에서 처리하는 계산 로직
복잡한 메타데이터가 붙은 수십 건의 경비를 모델이 머릿속으로 추적하고 합산하게 하는 대신, Claude는 산술과 집계 로직을 파이썬 코드에 위임했습니다. 덕분에 모델의 부담이 줄고, 계산이 정확해지며, 영수증 URL이나 가맹점 위치 같은 무관한 메타데이터는 모델의 컨텍스트에 아예 들어오지 않습니다.

---

## PTC를 언제 쓸까

PTC는 다음과 같은 경우에 가장 큰 효과를 냅니다.

- **메타데이터가 풍부한 대용량 데이터셋**을 필터링·파싱·집계해야 할 때(영수증 URL, 결재 라인, 가맹점 정보 등이 붙은 이번 경비 분석처럼)
- **순차적 의존 관계가 있어** 어떤 도구 호출이 이전 호출의 결과에 좌우될 때(표준 한도를 넘긴 직원만 예외 한도를 확인하는 경우처럼)
- 비슷한 대상에 대해 **여러 도구 호출**을 순차적으로 또는 반복문으로 해야 할 때(팀원마다 경비와 예산을 확인하는 경우처럼)
- **계산 로직**으로 모델의 컨텍스트를 통과해야 할 내용을 줄일 수 있을 때
- 사람의 감독 없이 프로그래매틱하게 반복 실행해도 **안전한 도구**일 때

## 마무리

이번 팀 경비 분석은 PTC의 강점을 잘 보여 주었습니다. **메타데이터가 풍부한 대용량 데이터셋을 다룰 때 컨텍스트 소비를 극적으로 줄이고**, **순차적 의존 관계가 있는 워크플로를 최적화**한다는 점입니다. Claude가 도구 호출을 오케스트레이션하고 결과를 프로그래매틱하게 처리하는 코드를 작성하게 함으로써, 정확도와 통찰의 품질은 유지하면서 상당한 토큰 절감을 이뤘습니다.

PTC는 메타데이터가 풍부한 대량 데이터 처리, 의존 관계가 있는 반복적 도구 호출, 또는 원시 도구 출력이 모델의 컨텍스트를 오염시킬 수 있는 상황에서 특히 가치가 큽니다.

## 다음 단계

이 패턴을 여러분의 사례에 적용해 보세요.
- 순차적 조회가 필요한 금융 데이터 분석과 보고
- 초기 스캔 결과에 따라 달라지는 다중 대상 상태 점검
- 메타데이터가 있는 대용량 파일 처리(CSV, JSON, XML 파싱)
- 후속 질의가 뒤따르는 데이터베이스 질의 결과 집계
- 초기 결과에 따른 조건 로직이 있는 배치 API 작업